In [1]:
import struct
import numpy as np

path = "../data/oracles/9s6d6c.dpairs2"

with open(path, "rb") as f:
    # Header (32 bytes)
    magic = f.read(8)
    assert magic == b"DPAIRS2\x00", f"Bad magic: {magic}"
    version, num_oop, num_ip, num_boundaries, num_iterations = struct.unpack("<5I", f.read(20))
    starting_pot = struct.unpack("<f", f.read(4))[0]

    print(f"Version: {version}")
    print(f"OOP hands: {num_oop}, IP hands: {num_ip}")
    print(f"Boundaries: {num_boundaries}")
    print(f"Iterations: {num_iterations}")
    print(f"Starting pot: {starting_pot}")
    print()

    # Read all iterations
    iterations = []
    for t in range(num_iterations):
        iteration, exploitability = struct.unpack("<If", f.read(8))
        conv_mode = struct.unpack("<I", f.read(4))[0] != 0

        boundary_cfvs = {}  # (boundary_idx, player) -> np.array
        for b in range(num_boundaries):
            for player in range(2):
                n = num_oop if player == 0 else num_ip
                cfv = np.frombuffer(f.read(n * 4), dtype=np.float32).copy()
                boundary_cfvs[(b, player)] = cfv

        iterations.append({
            "iteration": iteration,
            "exploitability": exploitability,
            "convergence_mode": conv_mode,
            "boundary_cfvs": boundary_cfvs,
        })

print(f"Loaded {len(iterations)} iterations")

Version: 2
OOP hands: 863, IP hands: 526
Boundaries: 25
Iterations: 150
Starting pot: 55.0

Loaded 150 iterations


In [2]:
# Iteration summary table
print(f"{'iter':>5} {'exploit%':>10} {'conv':>5}")
print("-" * 22)
for it in iterations:
    pct = it['exploitability'] / starting_pot * 100
    print(f"{it['iteration']:>5} {pct:>9.4f}% {str(it['convergence_mode']):>5}")

 iter   exploit%  conv
----------------------
    0   87.4478% False
    1   87.4478% False
    2   87.4478% False
    3   87.4478% False
    4   95.8700% False
    5   95.8700% False
    6   95.8700% False
    7   95.8700% False
    8   95.8700% False
    9   38.1089% False
   10   38.1089% False
   11   38.1089% False
   12   38.1089% False
   13   38.1089% False
   14   38.1089% False
   15   38.1089% False
   16   22.2772% False
   17   22.2772% False
   18   22.2772% False
   19   26.6559% False
   20   26.6559% False
   21   26.6559% False
   22   26.6559% False
   23   26.6559% False
   24   26.6559% False
   25   26.6559% False
   26   26.6559% False
   27   26.6559% False
   28   26.6559% False
   29    8.1582% False
   30    8.1582% False
   31    8.1582% False
   32    8.1582% False
   33    8.1582% False
   34    8.1582% False
   35    8.1582% False
   36    8.1582% False
   37    8.1582% False
   38    8.1582% False
   39    4.4645% False
   40    4.4645% False
   41    4.

In [3]:
# Inspect a specific boundary's CFV across iterations
boundary_idx = 0
player = 0  # 0=OOP, 1=IP
player_name = "OOP" if player == 0 else "IP"

print(f"Boundary {boundary_idx}, Player {player_name}")
print(f"CFV vector length: {len(iterations[0]['boundary_cfvs'][(boundary_idx, player)])}")
print()

print(f"{'iter':>5} {'min':>12} {'max':>12} {'mean':>12} {'std':>12} {'nonzero':>8}")
print("-" * 65)
for it in iterations:
    cfv = it['boundary_cfvs'][(boundary_idx, player)]
    nz = np.count_nonzero(cfv)
    print(f"{it['iteration']:>5} {cfv.min():>12.4f} {cfv.max():>12.4f} {cfv.mean():>12.4f} {cfv.std():>12.4f} {nz:>8}")

Boundary 0, Player OOP
CFV vector length: 863

 iter          min          max         mean          std  nonzero
-----------------------------------------------------------------
    0      -0.0109       0.0157      -0.0027       0.0073      863
    1      -0.0091       0.0072      -0.0053       0.0042      863
    2      -0.0283       0.0158      -0.0136       0.0116      863
    3      -0.0108       0.0124      -0.0047       0.0062      863
    4      -0.0187       0.0257      -0.0082       0.0122      863
    5      -0.0173       0.0166      -0.0103       0.0086      863
    6      -0.0215       0.0294      -0.0117       0.0126      863
    7      -0.0183       0.0334      -0.0099       0.0125      863
    8      -0.0125       0.0307      -0.0057       0.0103      863
    9      -0.0124       0.0294      -0.0062       0.0098      863
   10      -0.0117       0.0264      -0.0061       0.0089      863
   11      -0.0098       0.0284      -0.0041       0.0089      863
   12      -0.00

In [4]:
# How much do CFVs change between consecutive iterations?
boundary_idx = 0
player = 0

print(f"Boundary {boundary_idx}, Player {'OOP' if player == 0 else 'IP'} — iter-to-iter CFV delta")
print(f"{'iter':>5} {'max_delta':>12} {'mean_delta':>12} {'rmse':>12}")
print("-" * 45)
for i in range(1, len(iterations)):
    prev = iterations[i-1]['boundary_cfvs'][(boundary_idx, player)]
    curr = iterations[i]['boundary_cfvs'][(boundary_idx, player)]
    delta = np.abs(curr - prev)
    rmse = np.sqrt(np.mean((curr - prev) ** 2))
    print(f"{iterations[i]['iteration']:>5} {delta.max():>12.6f} {delta.mean():>12.6f} {rmse:>12.6f}")

Boundary 0, Player OOP — iter-to-iter CFV delta
 iter    max_delta   mean_delta         rmse
---------------------------------------------
    1     0.010180     0.003545     0.004834
    2     0.019878     0.010598     0.011547
    3     0.017738     0.009881     0.010538
    4     0.013904     0.006404     0.007058
    5     0.010349     0.003480     0.004888
    6     0.012835     0.003507     0.004267
    7     0.004011     0.001927     0.002100
    8     0.007498     0.004720     0.004826
    9     0.002012     0.000580     0.000820
   10     0.003061     0.000827     0.001093
   11     0.003265     0.002035     0.002085
   12     0.005113     0.001301     0.001858
   13     0.001622     0.000593     0.000725
   14     0.002194     0.000663     0.000940
   15     0.001612     0.000686     0.000791
   16     0.001774     0.001283     0.001323
   17     0.002578     0.000567     0.000895
   18     0.001221     0.000823     0.000835
   19     0.001212     0.000362     0.000506
   20 

In [5]:
# Overview of all boundaries at a specific iteration
iter_idx = len(iterations) - 1  # last iteration
it = iterations[iter_idx]

print(f"Iteration {it['iteration']} (exploit={it['exploitability']/starting_pot*100:.4f}%, conv={it['convergence_mode']})")
print()
print(f"{'boundary':>8} {'player':>7} {'min':>12} {'max':>12} {'mean':>12} {'std':>12}")
print("-" * 65)
for b in range(num_boundaries):
    for p in range(2):
        cfv = it['boundary_cfvs'][(b, p)]
        pname = "OOP" if p == 0 else "IP"
        print(f"{b:>8} {pname:>7} {cfv.min():>12.4f} {cfv.max():>12.4f} {cfv.mean():>12.4f} {cfv.std():>12.4f}")

Iteration 149 (exploit=0.4463%, conv=True)

boundary  player          min          max         mean          std
-----------------------------------------------------------------
       0     OOP      -0.0106       0.0363      -0.0023       0.0116
       0      IP      -0.0169       0.2258       0.0209       0.0485
       1     OOP      -0.0317       0.1007      -0.0099       0.0343
       1      IP      -0.0146       0.0593      -0.0021       0.0133
       2     OOP      -0.0216       0.0374      -0.0125       0.0141
       2      IP      -0.0446       0.1136      -0.0182       0.0355
       3     OOP      -0.0415       0.0450      -0.0182       0.0227
       3      IP      -0.0892       0.0665      -0.0506       0.0355
       4     OOP      -0.0028       0.0070      -0.0000       0.0027
       4      IP      -0.0000       0.0000      -0.0000       0.0000
       5     OOP      -0.0706       0.0613      -0.0385       0.0344
       5      IP      -0.0000       0.0000      -0.0000       